# AI-Enhanced SIEM Framework — Validation Notebook

**Author:** Srujan Subramanya Attota  
**Dataset:** [Microsoft GUIDE Dataset (Kaggle)](https://www.kaggle.com/datasets/Microsoft/microsoft-security-incident-prediction)

This notebook validates the proposed AI-Enhanced SIEM Framework across **four experiments**, each corresponding to a stage of the incident response pipeline:

| Experiment | Stage | What is validated |
|---|---|---|
| 1 | Incident Detection | XGBoost classification accuracy on `IncidentGrade` labels |
| 2 | Alert Prioritisation | Hybrid risk score (ML + context) vs ML-only true-positive retrieval |
| 3 | Investigation Workload | Incident-grouping reduces analyst review items |
| 4 | Response Recommendation | Automated playbook mapping by attack category and incident grade |

---

## How to Run

### 1. Install dependencies
```bash
pip install pandas numpy scikit-learn xgboost matplotlib seaborn openpyxl
```

### 2. Download the dataset
Download `GUIDE_Train.csv` from [Kaggle](https://www.kaggle.com/datasets/Microsoft/microsoft-security-incident-prediction) and place it in the **same folder as this notebook**.

### 3. Run all cells
**Kernel → Restart & Run All**

---

## Files Generated

After running all cells, the following files are saved in the working directory:

| File | Type | Content |
|------|------|---------|
| `table_detection_performance.csv` | Table | Accuracy, Precision, Recall, F1 |
| `table_prioritisation_comparison.csv` | Table | TP rate at Top 10/20/25/30% |
| `table_investigation_workload.csv` | Table | Alert rows vs incident groups |
| `table_response_summary.csv` | Table | Response actions per IncidentGrade |
| `table_response_action_distribution.csv` | Table | Top 15 response actions issued |
| `SIEM_Framework_Results_Tables.xlsx` | Excel | All tables in one workbook |
| `figure_normalised_confusion_matrix.png` | Figure 1 | Row-normalised confusion matrix heatmap |
| `figure_prioritisation_comparison.png` | Figure 2 | Line chart — current vs proposed prioritisation |
| `figure_investigation_workload.png` | Figure 3 | Bar chart — review item count comparison |
| `figure_response_summary.png` | Figure 4 | Bar chart — records per IncidentGrade |


---
## 1. Environment Check

Confirms the working directory and lists files present. Run this first to verify `GUIDE_Train.csv` is in the right place before loading data.

In [ ]:
import os
import pandas as pd

print("Current folder:", os.getcwd())
print("Files here:", os.listdir("."))

---
## 2. Load Dataset

Loads the first **300,000 rows** of `GUIDE_Train.csv`.

The GUIDE dataset is Microsoft's security telemetry dataset. Key columns used in this notebook:

| Column | Description |
|--------|-------------|
| `IncidentGrade` | Target label — `TruePositive`, `BenignPositive`, `FalsePositive` |
| `IncidentId` | Groups multiple alert rows into one incident |
| `Category` | Attack category (e.g. Malware, Exfiltration, CredentialAccess) |
| `EvidenceRole` | Whether the entity was impacted or compromised |
| `SuspicionLevel` | Analyst-assigned suspicion level |
| `ThreatFamily` | Known threat family name if available |
| `MitreTechniques` | MITRE ATT&CK technique identifier |
| `LastVerdict` | Most recent automated verdict |

> To load more data, increase `nrows`. The full dataset has ~1 million rows.

In [ ]:
import pandas as pd

df = pd.read_csv("GUIDE_Train.csv", nrows=300000)

print(df.shape)
print(df.columns.tolist())
df.head()

---
## 3. Class Distribution

Checks how many records fall into each `IncidentGrade` category.  
This is important before training — if one class has far fewer records, the model may be biased.

**Expected output:** Counts for `TruePositive`, `BenignPositive`, and `FalsePositive`.

In [ ]:
df["IncidentGrade"].value_counts(dropna=False)

---
## 4. Feature Engineering and Train / Test Split

**Steps performed in this cell:**

1. **Drop null target rows** — rows without an `IncidentGrade` label are removed.
2. **Save `raw_df`** — an unencoded copy is kept for the investigation and response experiments that need the original string values (e.g. `Category`, `IncidentId`).
3. **Remove ID columns** — `IncidentId`, `AlertId`, `OrgId` are dropped to prevent data leakage (the model should not learn identifiers).
4. **Label-encode all categoricals** — both the target and all string feature columns are integer-encoded.
5. **Fill missing values** — remaining `NaN` values are filled with `0`.
6. **80/20 stratified split** — stratification ensures each class is proportionally represented in both train and test sets.

`X_train`, `X_test`, `y_train`, `y_test` are used throughout the rest of the notebook.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Remove rows where IncidentGrade is missing
df = df.dropna(subset=["IncidentGrade"]).copy()

# Keep raw copy for investigation and response simulations
raw_df = df.copy()

target = "IncidentGrade"

# Drop ID columns so model does not learn identifiers
drop_cols = ["IncidentId", "AlertId", "OrgId"]

X = df.drop(columns=[target] + [c for c in drop_cols if c in df.columns], errors="ignore")
y = df[target]

# Encode target
target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(y.astype(str))

# Encode categorical columns
for col in X.select_dtypes(include=["object"]).columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

# Fill missing values
X = X.fillna(0)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("Training data:", X_train.shape)
print("Testing data:",  X_test.shape)
print("Target classes:", target_encoder.classes_)

---
## Experiment 1 — Incident Detection Performance

### 5. Train XGBoost Classifier

Trains an **XGBoost** multi-class classifier to predict `IncidentGrade`.  
This represents the **Current AI-SIEM** detection baseline.

**Model configuration:**

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `n_estimators` | 200 | Sufficient rounds for convergence on 240k training rows |
| `max_depth` | 6 | Balances model complexity and overfitting |
| `learning_rate` | 0.1 | Standard starting point for XGBoost |
| `eval_metric` | mlogloss | Multi-class log loss — appropriate for 3-class problem |
| `random_state` | 42 | Reproducibility |

**Output:** Full `classification_report` (per-class precision, recall, F1) plus a summary dict.  

> **Generates:** `table_detection_performance.csv`

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

current_ai_model = XGBClassifier(
    eval_metric="mlogloss",
    random_state=42,
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1
)

current_ai_model.fit(X_train, y_train)

current_preds = current_ai_model.predict(X_test)
current_probs = current_ai_model.predict_proba(X_test)

print(classification_report(y_test, current_preds, target_names=target_encoder.classes_))

current_results = {
    "Approach": "Current AI-SIEM",
    "Accuracy": accuracy_score(y_test, current_preds),
    "Macro Precision": precision_score(y_test, current_preds, average="macro"),
    "Macro Recall":    recall_score(y_test, current_preds, average="macro"),
    "Macro F1":        f1_score(y_test, current_preds, average="macro")
}

current_results

---
### 6. Initial Context Score (v1 — encoded features)

An early version of the context score built from **encoded** feature columns using keyword matching on column names.  
Weights are assigned based on how strongly each column type correlates with incident severity:

| Column keyword | Weight |
|----------------|--------|
| `severity` | 0.30 |
| `priority` | 0.25 |
| `category` | 0.15 |
| `entity` | 0.15 |
| `detector` | 0.10 |
| `evidence` | 0.05 |

This version is superseded by the **v3 context score** (Cell 10) which uses the raw unencoded data for more meaningful signal.  
The output `risk_df` is an intermediate preview — it is not used directly in the final comparison.

In [ ]:
ml_confidence = current_probs.max(axis=1)

X_test_copy = X_test.copy()

context_score = pd.Series(0, index=X_test_copy.index, dtype=float)

for col in X_test_copy.columns:
    col_lower = col.lower()

    if "severity" in col_lower:
        context_score += X_test_copy[col] * 0.30

    if "priority" in col_lower:
        context_score += X_test_copy[col] * 0.25

    if "category" in col_lower:
        context_score += X_test_copy[col] * 0.15

    if "entity" in col_lower:
        context_score += X_test_copy[col] * 0.15

    if "detector" in col_lower:
        context_score += X_test_copy[col] * 0.10

    if "evidence" in col_lower:
        context_score += X_test_copy[col] * 0.05

# Normalise context score to [0, 1]
context_score = (context_score - context_score.min()) / (
    context_score.max() - context_score.min() + 1e-9
)

# Proposed hybrid risk score (v1)
hybrid_risk_score = (0.70 * ml_confidence) + (0.30 * context_score.values)

risk_df = pd.DataFrame({
    "Actual": target_encoder.inverse_transform(y_test),
    "Predicted": target_encoder.inverse_transform(current_preds),
    "MLConfidence": ml_confidence,
    "ContextScore": context_score.values,
    "HybridRiskScore": hybrid_risk_score
})

risk_df.head()

---
### 7. Identify TruePositive Class Index

The `LabelEncoder` assigns an integer index to each class in alphabetical order.  
We need the index of `TruePositive` to extract the correct probability column from `predict_proba` output.

In [ ]:
print("Encoded class order:", target_encoder.classes_)

tp_index = list(target_encoder.classes_).index("TruePositive")
print("TruePositive is at index:", tp_index)

---
### 8. Extract TruePositive Probability per Row

`current_probs[:, tp_index]` gives the model's estimated probability that each test record is a `TruePositive`.  
This is the **Current AI-SIEM** prioritisation signal — ML confidence alone with no contextual adjustment.

`risk_df_tp` is a preview DataFrame showing actual label, predicted label, and TP probability for each test row.

In [ ]:
# Probability that each row is TruePositive
tp_probability = current_probs[:, tp_index]

risk_df_tp = pd.DataFrame({
    "Actual": target_encoder.inverse_transform(y_test),
    "Predicted": target_encoder.inverse_transform(current_preds),
    "TruePositiveProbability": tp_probability
})

risk_df_tp.head()

---
### 9. Baseline — Current AI-SIEM True Positive Rate at Top 25%

Evaluates how many of the top 25% highest-probability alerts (ranked by `TruePositiveProbability`) are actually `TruePositive`.  
This is the **baseline** that the proposed hybrid score must beat.

In [ ]:
top_25_count = int(len(risk_df_tp) * 0.25)

current_tp_rate_corrected = (
    risk_df_tp.nlargest(top_25_count, "TruePositiveProbability")["Actual"]
    .eq("TruePositive")
    .mean()
)

print(f"Current AI-SIEM — Top 25% True Positive Rate: {current_tp_rate_corrected:.4f}")

---
## Experiment 2 — Alert Prioritisation with Hybrid Risk Score

### 10. Build Context-Aware Risk Score (v3 — raw features)

This is the **core contribution** of the proposed framework for the prioritisation experiment.  
Instead of ranking alerts purely by ML confidence, a **context-aware risk score** is added using the raw (unencoded) feature columns.

**Context score components and weights:**

| Feature | Signal used | Weight | Rationale |
|---------|------------|--------|-----------|
| `Category` | Is it a high-risk attack type? | 0.30 | Direct threat indicator — highest weight |
| `EvidenceRole` | Contains Impacted / Compromised? | 0.20 | Confirms asset was affected |
| `SuspicionLevel` | Field is populated? | 0.15 | Analyst-assigned suspicion present |
| `ThreatFamily` | Field is populated? | 0.15 | Links to known threat family |
| `MitreTechniques` | Field is populated? | 0.10 | ATT&CK technique recorded |
| `LastVerdict` | Contains Malicious / Suspicious? | 0.10 | Historical verdict supports escalation |

**High-risk categories:** CommandAndControl, Exfiltration, CredentialAccess, Malware, Persistence, PrivilegeEscalation, Execution, LateralMovement, Impact

**Hybrid risk score formula:**  
```
hybrid_score = (0.80 × TruePositive_probability) + (0.20 × context_score)
```  
The 80/20 split keeps ML as the dominant signal while adding meaningful contextual uplift.

In [ ]:
# Align raw test data with encoded test indices
raw_test = raw_df.loc[X_test.index].copy()

context_score_v3 = pd.Series(0, index=raw_test.index, dtype=float)

# High-risk categories based on common security incident types
high_risk_categories = [
    "CommandAndControl", "Exfiltration", "CredentialAccess",
    "Malware", "Persistence", "PrivilegeEscalation",
    "Execution", "LateralMovement", "Impact"
]

if "Category" in raw_test.columns:
    context_score_v3 += raw_test["Category"].isin(high_risk_categories).astype(int) * 0.30

if "EvidenceRole" in raw_test.columns:
    context_score_v3 += raw_test["EvidenceRole"].astype(str).str.contains(
        "Impacted|Compromised|Related", case=False, na=False
    ).astype(int) * 0.20

if "SuspicionLevel" in raw_test.columns:
    context_score_v3 += raw_test["SuspicionLevel"].notna().astype(int) * 0.15

if "ThreatFamily" in raw_test.columns:
    context_score_v3 += raw_test["ThreatFamily"].notna().astype(int) * 0.15

if "MitreTechniques" in raw_test.columns:
    context_score_v3 += raw_test["MitreTechniques"].notna().astype(int) * 0.10

if "LastVerdict" in raw_test.columns:
    context_score_v3 += raw_test["LastVerdict"].astype(str).str.contains(
        "Malicious|Suspicious", case=False, na=False
    ).astype(int) * 0.10

# Normalise context score to [0, 1]
context_score_v3 = (context_score_v3 - context_score_v3.min()) / (
    context_score_v3.max() - context_score_v3.min() + 1e-9
)

# Proposed framework hybrid score
hybrid_risk_score_v3 = (0.80 * tp_probability) + (0.20 * context_score_v3.values)

risk_df_v3 = pd.DataFrame({
    "Actual": target_encoder.inverse_transform(y_test),
    "Predicted": target_encoder.inverse_transform(current_preds),
    "TruePositiveProbability": tp_probability,
    "ContextScore": context_score_v3.values,
    "HybridRiskScore": hybrid_risk_score_v3
})

risk_df_v3.head()

---
### 11. Head-to-Head Comparison at Top 25%

Directly compares the two approaches at the Top 25% threshold.

**`prioritisation_results_v3`** is a two-row summary table showing:
- **Current AI-SIEM:** ranks by ML probability only
- **Proposed Framework:** ranks by hybrid score (ML + context)

A higher True Positive Rate means analysts encounter more real incidents and fewer false alarms in the same review budget.

In [ ]:
top_25_count = int(len(risk_df_v3) * 0.25)

current_top_tp_rate = (
    risk_df_v3.nlargest(top_25_count, "TruePositiveProbability")["Actual"]
    .eq("TruePositive")
    .mean()
)

proposed_top_tp_rate = (
    risk_df_v3.nlargest(top_25_count, "HybridRiskScore")["Actual"]
    .eq("TruePositive")
    .mean()
)

prioritisation_results_v3 = pd.DataFrame([
    {
        "Approach": "Current AI-SIEM",
        "Prioritisation Method": "TruePositive probability only",
        "Top 25% True Positive Rate": current_top_tp_rate
    },
    {
        "Approach": "Proposed Framework",
        "Prioritisation Method": "TruePositive probability + context-aware risk score",
        "Top 25% True Positive Rate": proposed_top_tp_rate
    }
])

prioritisation_results_v3

---
### 12. Multi-Threshold Prioritisation Comparison

**Table: Alert Prioritisation — True Positive Rate at Multiple Review Thresholds**

Evaluates both approaches at four review budgets: Top 10%, 20%, 25%, and 30%.  
The `Improvement` column shows the absolute gain of the proposed framework over the current approach at each threshold.

> **Generates:** `table_prioritisation_comparison.csv`  
> **Feeds into:** Figure 2 (line chart)

In [ ]:
rows = []

for top_percent in [0.10, 0.20, 0.25, 0.30]:
    top_count = int(len(risk_df_v3) * top_percent)

    current_rate = (
        risk_df_v3.nlargest(top_count, "TruePositiveProbability")["Actual"]
        .eq("TruePositive")
        .mean()
    )

    proposed_rate = (
        risk_df_v3.nlargest(top_count, "HybridRiskScore")["Actual"]
        .eq("TruePositive")
        .mean()
    )

    rows.append({
        "Top Percentage": f"Top {int(top_percent*100)}%",
        "Current AI-SIEM": current_rate,
        "Proposed Framework": proposed_rate,
        "Improvement": proposed_rate - current_rate
    })

top_percentage_results = pd.DataFrame(rows)
top_percentage_results

---
## Experiment 3 — Investigation Workload Reduction

### 13. Alert Rows vs Incident Groups

**Table: Investigation Workload Comparison**

The Current AI-SIEM presents analysts with every individual alert/evidence row.  
The Proposed Framework groups rows by `IncidentId`, so analysts review one entry per incident rather than one per alert.

**Workload Reduction formula:**  
```
Workload Reduction % = (1 - unique_incidents / total_rows) × 100
```

> **Generates:** `table_investigation_workload.csv`  
> **Feeds into:** Figure 3 (bar chart)

In [ ]:
# Investigation support experiment

total_rows = len(raw_df)

if "IncidentId" in raw_df.columns:
    total_incidents = raw_df["IncidentId"].nunique()
    workload_reduction = (1 - total_incidents / total_rows) * 100

    investigation_results = pd.DataFrame([
        {
            "Approach": "Current AI-SIEM",
            "Review Unit": "Individual evidence/alert rows",
            "Review Items": total_rows,
            "Workload Reduction": "Baseline"
        },
        {
            "Approach": "Proposed Framework",
            "Review Unit": "Incident-level grouped view",
            "Review Items": total_incidents,
            "Workload Reduction": f"{workload_reduction:.2f}%"
        }
    ])

investigation_results

---
### 14. Figure 3 — Investigation Workload Bar Chart

Visualises the difference in review item count between the two approaches.  
The lower bar (Proposed Framework) represents the workload reduction achieved by incident-level grouping.

> **Saves:** `figure_investigation_workload.png` (300 dpi)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.bar(
    investigation_results["Approach"],
    investigation_results["Review Items"]
)
plt.ylabel("Number of Review Items")
plt.title("Investigation Workload — Current AI-SIEM vs Proposed Framework")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("figure_investigation_workload.png", dpi=300)
plt.show()
print("Saved: figure_investigation_workload.png")

---
## Experiment 4 — Automated Response Recommendation

### 15. Playbook Mapping by Category × IncidentGrade

**Table: Automated Response Recommendations (sample)**

Maps each alert to a recommended response action using a decision matrix based on `Category` and `IncidentGrade`.

**Decision logic:**

| IncidentGrade | Action prefix |
|---------------|---------------|
| `TruePositive` | `Immediate action:` + category-specific containment |
| `BenignPositive` | `Validate and monitor:` + category-specific guidance |
| `FalsePositive` | `No containment; tune detection logic` |
| Unknown | `Analyst review required` |

**Categories with specific playbooks:** Malware, Phishing, CredentialAccess, CommandAndControl, Exfiltration, Reconnaissance, Persistence, PrivilegeEscalation, Execution, InitialAccess, LateralMovement

In [ ]:
response_map = {
    "Malware":             "Isolate endpoint; run malware scan; collect forensic evidence",
    "Phishing":            "Disable account; reset password; review MFA and mailbox rules",
    "CredentialAccess":    "Reset credentials; revoke sessions; enforce MFA review",
    "CommandAndControl":   "Block domain/IP; isolate endpoint; investigate process tree",
    "Exfiltration":        "Block outbound channel; isolate host; review accessed data",
    "Reconnaissance":      "Monitor source; block repeated scanning; increase logging",
    "Persistence":         "Remove persistence mechanism; isolate host; review autoruns",
    "PrivilegeEscalation": "Disable affected account; review admin group changes",
    "Execution":           "Isolate endpoint if confirmed; collect process and command-line evidence",
    "InitialAccess":       "Validate access source; reset credentials if account compromise is suspected",
    "LateralMovement":     "Isolate affected hosts; review authentication paths and admin activity"
}

def recommend_response(row):
    category = str(row.get("Category", "Unknown"))
    grade    = str(row.get("IncidentGrade", "Unknown"))
    base     = response_map.get(category, "Analyst review required")

    if grade == "TruePositive":
        return "Immediate action: " + base
    elif grade == "BenignPositive":
        return "Validate and monitor: " + base
    elif grade == "FalsePositive":
        return "No containment; tune detection logic"
    else:
        return "Analyst review required"

raw_df["RecommendedResponse"] = raw_df.apply(recommend_response, axis=1)

# Preview first 20 rows
raw_df[["Category", "IncidentGrade", "RecommendedResponse"]].head(20)

---
### 16. Response Summary by IncidentGrade

**Table: Response Recommendation Summary**

Groups records by `IncidentGrade` and shows:
- `TotalRecords` — how many alerts received each grade
- `UniqueResponses` — how many distinct response actions were issued for that grade

> **Generates:** `table_response_summary.csv`  
> **Feeds into:** Figure 4 (bar chart)

In [ ]:
response_summary = raw_df.groupby(["IncidentGrade"]).agg(
    TotalRecords=("IncidentGrade", "count"),
    UniqueResponses=("RecommendedResponse", "nunique")
).reset_index()

response_summary

---
### 17. Top 15 Response Actions

**Table: Response Action Distribution**

Shows the 15 most frequently issued response actions across the entire dataset.  
High counts for `Immediate action:` entries indicate a large proportion of true positive incidents.

> **Generates:** `table_response_action_distribution.csv`

In [ ]:
response_action_distribution = raw_df["RecommendedResponse"].value_counts().head(15).reset_index()
response_action_distribution.columns = ["Recommended Response", "Count"]

response_action_distribution

---
## Save All Result Tables

### 18. Export to CSV and Excel

Saves every result table as an individual CSV and as a single multi-sheet Excel workbook.  
All files are written to the current working directory (same folder as this notebook).

| Sheet name | Content |
|-----------|---------|
| Detection Performance | Accuracy, Precision, Recall, F1 |
| Prioritisation | TP rate at Top 10/20/25/30% for both approaches |
| Investigation | Alert row count vs incident group count |
| Response Summary | Records and unique responses per IncidentGrade |
| Response Distribution | Top 15 most-issued response actions |

In [ ]:
# Individual CSVs
pd.DataFrame([current_results]).to_csv("table_detection_performance.csv", index=False)
top_percentage_results.to_csv("table_prioritisation_comparison.csv", index=False)
investigation_results.to_csv("table_investigation_workload.csv", index=False)
response_summary.to_csv("table_response_summary.csv", index=False)
response_action_distribution.to_csv("table_response_action_distribution.csv", index=False)

# Combined Excel workbook
with pd.ExcelWriter("SIEM_Framework_Results_Tables.xlsx") as writer:
    pd.DataFrame([current_results]).to_excel(writer, sheet_name="Detection Performance", index=False)
    top_percentage_results.to_excel(writer, sheet_name="Prioritisation", index=False)
    investigation_results.to_excel(writer, sheet_name="Investigation", index=False)
    response_summary.to_excel(writer, sheet_name="Response Summary", index=False)
    response_action_distribution.to_excel(writer, sheet_name="Response Distribution", index=False)

print("All tables saved.")
for f in [
    "table_detection_performance.csv",
    "table_prioritisation_comparison.csv",
    "table_investigation_workload.csv",
    "table_response_summary.csv",
    "table_response_action_distribution.csv",
    "SIEM_Framework_Results_Tables.xlsx"
]:
    print(" -", f)

---
## Figures

### 19. Figure 2 — Alert Prioritisation Line Chart

**Figure 2: Current AI-SIEM vs Proposed Framework — True Positive Rate at Multiple Thresholds**

Each point on the x-axis represents the top N% of alerts reviewed (ranked by the respective scoring method).  
The y-axis shows what fraction of those reviewed alerts are genuine `TruePositive` incidents.

- A **higher line** means the approach surfaces more real incidents within the same review budget.
- The gap between the two lines shows the improvement contributed by the context-aware hybrid score.

> **Saves:** `figure_prioritisation_comparison.png` (300 dpi)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(
    top_percentage_results["Top Percentage"],
    top_percentage_results["Current AI-SIEM"],
    marker="o", label="Current AI-SIEM"
)
plt.plot(
    top_percentage_results["Top Percentage"],
    top_percentage_results["Proposed Framework"],
    marker="o", label="Proposed Framework"
)
plt.ylabel("True Positive Rate")
plt.xlabel("Top Review Group")
plt.title("Alert Prioritisation — Current AI-SIEM vs Proposed Framework")
plt.legend()
plt.tight_layout()
plt.savefig("figure_prioritisation_comparison.png", dpi=300)
plt.show()
print("Saved: figure_prioritisation_comparison.png")

---
### 20. Figure 3 — Investigation Workload Bar Chart (duplicate save)

Second save of the workload chart for completeness.  
This version is identical to Figure 3 generated in Cell 14.

> **Saves:** `figure_investigation_workload.png` (300 dpi)

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(investigation_results["Approach"], investigation_results["Review Items"])
plt.ylabel("Number of Review Items")
plt.title("Investigation Workload Comparison")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("figure_investigation_workload.png", dpi=300)
plt.show()
print("Saved: figure_investigation_workload.png")

---
### 21. Figure 4 — Response Recommendation by Incident Grade

**Figure 4: Response Recommendation Volume by Incident Grade**

Bar chart showing how many records in the dataset received each `IncidentGrade`.  
Taller bars indicate higher volumes of that incident type in the GUIDE dataset.

> **Saves:** `figure_response_summary.png` (300 dpi)

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(response_summary["IncidentGrade"], response_summary["TotalRecords"])
plt.ylabel("Number of Records")
plt.xlabel("Incident Grade")
plt.title("Response Recommendation Volume by Incident Grade")
plt.tight_layout()
plt.savefig("figure_response_summary.png", dpi=300)
plt.show()
print("Saved: figure_response_summary.png")

---
### 22. Figure 1 — Normalised Confusion Matrix

**Figure 1: Normalised Confusion Matrix — Current AI-SIEM Model (%)**

Each cell shows the **percentage** of actual class (row) predicted as each class (column).  
Row sums to 100%. The diagonal shows correct classification rates.

- High diagonal values = model predicts that class well.
- Off-diagonal values = misclassification patterns (e.g. FalsePositives misclassified as BenignPositive).

> **Saves:** `figure_normalised_confusion_matrix.png` (300 dpi)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

cm = confusion_matrix(y_test, current_preds)

# Convert to row-normalised percentages
cm_percent = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis] * 100

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm_percent,
    annot=True,
    fmt=".1f",
    cmap="Blues",
    xticklabels=target_encoder.classes_,
    yticklabels=target_encoder.classes_
)

plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.title("Normalised Confusion Matrix — Current AI-SIEM Model (%)")
plt.tight_layout()
plt.savefig("figure_normalised_confusion_matrix.png", dpi=300)
plt.show()
print("Saved: figure_normalised_confusion_matrix.png")

---
### 23. Per-Class Accuracy Table

**Table: Correct Classification Rate per Incident Grade**

Extracts the diagonal of the confusion matrix to show accuracy per class.  
Useful for reporting which incident types the model handles well vs. which need improvement.

In [ ]:
cm = confusion_matrix(y_test, current_preds)
classes = target_encoder.classes_

per_class_accuracy = cm.diagonal() / cm.sum(axis=1)

class_accuracy_table = pd.DataFrame({
    "Incident Grade": classes,
    "Correct Classification Rate": per_class_accuracy
})

class_accuracy_table